# Album catalog enrichment

Enrich `albums.csv` from two sources, then join Last.fm fields onto embedding recommendations.

**Sources**
1. `album_catalog_enriched.parquet` — review-source metadata, Pitchfork genre/ratings, MusicBrainz IDs, release info, tags/genre (`artist_norm` / `album_norm` join).
2. `lastfm_recommendations_all_top_listener.csv` — listener counts and top tags from the batch run (`album_id` for seeds; normalized artist/album for rec-only albums).

**Workflow (re-run while the Last.fm batch is still streaming):**
1. `lastfm-bench/lastfm_batch.ipynb` — resume until the `all` subset finishes.
2. This notebook — refresh `albums.csv` and `final_recs.parquet` from whatever is available so far.

List-like parquet fields (`sources`, `mb_tags`) are written to CSV as semicolon-separated strings.

In [31]:
import re
import unicodedata
from pathlib import Path

import pandas as pd

DATA_DIR = Path("../datasets")
ALBUMS_PATH = DATA_DIR / "albums.csv"
CATALOG_ENRICHED_PATH = DATA_DIR / "album_catalog_enriched.parquet"
LASTFM_PATH = DATA_DIR / "lastfm_recommendations_all_top_listener.csv"
RECS_PATH = DATA_DIR / "recommendations_album_level_avg_embeddings.parquet"
RECS_OUT_PATH = DATA_DIR / "final_recs.parquet"

LISTENER_COL = "lastfm_listeners"
TAGS_COL = "lastfm_tags"

CATALOG_META_COLS = [
    "genre_pitchfork",
    "rating_mean",
    "rating_pf",
    "sources",
    "n_sources",
    "published_on_min",
    "published_on_max",
    "ra_recommends",
    "mb_release_group_mbid",
    "mb_artist_mbid",
    "release_year",
    "release_type",
    "artist_type",
    "artist_area",
    "label",
    "catalog_number",
    "mb_tags",
    "mb_genre",
]
ARRAY_COLS = {"sources", "mb_tags"}

In [32]:
def normalize(s):
    if pd.isna(s):
        return ""
    s = str(s).strip().lower()
    for ch in ("\u2019", "\u2018", "\u201b", "\u2032", "`", "\u00b4"):
        s = s.replace(ch, "'")
    s = s.replace("\u2026", "...").replace("\u00b7", " ")
    s = unicodedata.normalize("NFKD", s)
    s = "".join(c for c in s if not unicodedata.combining(c))
    return re.sub(r"\s+", " ", s)


def add_norm_keys(df, artist_col, album_col):
    out = df.copy()
    out["artist_norm"] = out[artist_col].map(normalize)
    out["album_norm"] = out[album_col].map(normalize)
    out.loc[out["artist_norm"].isin(["various", "va"]), "artist_norm"] = "various artists"
    return out


def serialize_array_col(val):
    """Parquet list/array columns -> semicolon-separated CSV strings."""
    if val is None or (isinstance(val, float) and pd.isna(val)):
        return pd.NA
    if isinstance(val, (list, tuple)):
        items = val
    elif hasattr(val, "tolist"):
        items = val.tolist()
    else:
        text = str(val).strip()
        return pd.NA if text in ("", "None", "nan", "[]") else text
    cleaned = [str(item).strip() for item in items if str(item).strip()]
    return ";".join(cleaned) if cleaned else pd.NA

## Load batch output

In [33]:
lastfm = pd.read_csv(LASTFM_PATH, low_memory=False)
for col in ("seed_listeners", "rec_listeners"):
    lastfm[col] = pd.to_numeric(lastfm[col], errors="coerce").astype("Int64")
for col in ("seed_tags", "rec_tags"):
    lastfm[col] = lastfm[col].astype("string")

ok = lastfm.loc[lastfm["status"] == "ok"].copy()
print(f"Batch rows: {len(lastfm):,}")
print(f"OK rows:    {len(ok):,}  |  query albums: {ok['album_id'].nunique():,}")
print(
    f"Seed listeners: {ok['seed_listeners'].notna().mean():.1%}  |  "
    f"seed tags: {ok['seed_tags'].notna().mean():.1%}  |  "
    f"rec tags: {ok['rec_tags'].notna().mean():.1%}"
)

Batch rows: 27,954
OK rows:    24,941  |  query albums: 6,027
Seed listeners: 100.0%  |  seed tags: 68.3%  |  rec tags: 63.2%


## Aggregate per album

Build one metadata row per catalog album: seeds keyed by `album_id`, rec-only hits keyed by normalized artist/album.

In [34]:
def best_meta_rows(frame, group_cols, listener_col, tags_col):
    """One row per group; prefer the highest listener count."""
    ordered = frame.sort_values(listener_col, ascending=False, na_position="last")
    return ordered.drop_duplicates(group_cols, keep="first")


seed_rows = ok.loc[:, ["album_id", "seed_listeners", "seed_tags"]].rename(
    columns={"seed_listeners": LISTENER_COL, "seed_tags": TAGS_COL}
)
seed_meta = best_meta_rows(seed_rows, ["album_id"], LISTENER_COL, TAGS_COL)

rec_rows = add_norm_keys(
    ok.loc[:, ["rec_artist", "rec_album", "rec_listeners", "rec_tags"]],
    "rec_artist",
    "rec_album",
).rename(columns={"rec_listeners": LISTENER_COL, "rec_tags": TAGS_COL})
rec_meta = best_meta_rows(rec_rows, ["artist_norm", "album_norm"], LISTENER_COL, TAGS_COL)

print(f"Seed metadata rows: {len(seed_meta):,}")
print(f"Rec metadata keys:  {len(rec_meta):,}")

Seed metadata rows: 6,027
Rec metadata keys:  18,013


## Load `album_catalog_enriched.parquet`

Joined on `artist_norm` / `album_norm`. The enriched catalog is larger than `albums.csv` (includes RA-only albums), so not every review-catalog album will match.

In [35]:
catalog_meta = pd.read_parquet(CATALOG_ENRICHED_PATH)
catalog_meta = catalog_meta.drop(columns=["artist", "album"], errors="ignore")
for col in ARRAY_COLS:
    catalog_meta[col] = catalog_meta[col].map(serialize_array_col).astype("string")

dupes = catalog_meta.duplicated(["artist_norm", "album_norm"], keep=False).sum()
if dupes:
    raise ValueError(f"Expected unique norm keys in {CATALOG_ENRICHED_PATH.name}; found {dupes} duplicates")

print(f"Enriched catalog rows: {len(catalog_meta):,}")
for col in CATALOG_META_COLS:
    print(f"  {col}: {catalog_meta[col].notna().mean():.1%}")

Enriched catalog rows: 44,716
  genre_pitchfork: 76.5%
  rating_mean: 84.9%
  rating_pf: 51.0%
  sources: 100.0%
  n_sources: 100.0%
  published_on_min: 100.0%
  published_on_max: 100.0%
  ra_recommends: 100.0%
  mb_release_group_mbid: 63.9%
  mb_artist_mbid: 63.9%
  release_year: 63.7%
  release_type: 63.3%
  artist_type: 62.7%
  artist_area: 57.2%
  label: 62.0%
  catalog_number: 56.0%
  mb_tags: 54.0%
  mb_genre: 53.3%


## Merge into `albums.csv`

Catalog metadata first (norm-key join), then Last.fm listeners/tags (seed `album_id`, then rec norm-key fill).

In [36]:
albums = pd.read_csv(ALBUMS_PATH)
albums = add_norm_keys(albums, "artist", "album")

drop_cols = set(CATALOG_META_COLS + [LISTENER_COL, TAGS_COL, "artist_norm", "album_norm"])
base_cols = [c for c in albums.columns if c not in drop_cols]
albums = albums[base_cols + ["artist_norm", "album_norm"]]

albums = albums.merge(catalog_meta, on=["artist_norm", "album_norm"], how="left")

albums = albums.merge(seed_meta[["album_id", LISTENER_COL, TAGS_COL]], on="album_id", how="left")
rec_fill = albums.merge(
    rec_meta[["artist_norm", "album_norm", LISTENER_COL, TAGS_COL]],
    on=["artist_norm", "album_norm"],
    how="left",
    suffixes=("", "_rec"),
)
albums[LISTENER_COL] = albums[LISTENER_COL].fillna(rec_fill[f"{LISTENER_COL}_rec"])
albums[TAGS_COL] = albums[TAGS_COL].fillna(rec_fill[f"{TAGS_COL}_rec"])

n_catalog = albums["mb_release_group_mbid"].notna().sum()
n_listeners = albums[LISTENER_COL].notna().sum()
n_tags = albums[TAGS_COL].notna().sum()
print(f"Catalog albums: {len(albums):,}")
print(f"With catalog metadata: {n_catalog:,} ({n_catalog / len(albums):.1%})")
print(f"With listeners:        {n_listeners:,} ({n_listeners / len(albums):.1%})")
print(f"With Last.fm tags:     {n_tags:,} ({n_tags / len(albums):.1%})")

out_albums = albums.drop(columns=["artist_norm", "album_norm"])
out_albums.to_csv(ALBUMS_PATH, index=False)
print(f"Saved {ALBUMS_PATH}")
out_albums.head()

Catalog albums: 30,516
With catalog metadata: 22,685 (74.3%)
With listeners:        8,814 (28.9%)
With Last.fm tags:     6,431 (21.1%)
Saved ../datasets/albums.csv


,album_id,artist,album,review_count,genre_pitchfork,rating_mean,rating_pf,sources,n_sources,published_on_min,...,release_year,release_type,artist_type,artist_area,label,catalog_number,mb_tags,mb_genre,lastfm_listeners,lastfm_tags
0,0,!!!,As If,1,Rock,3.45,3.45,pitchfork,1.0,2015-10-21,...,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,<NA>,<NA>
1,1,!!!,"Jamie, My Intentions Are Bass EP",1,Rock,3.40,3.40,pitchfork,1.0,2010-11-01,...,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,<NA>,<NA>
2,2,!!!,Louden Up Now,1,Rock,3.50,3.50,pitchfork,1.0,2004-06-07,...,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,195828,<NA>
3,3,!!!,Myth Takes,2,Rock,4.00,4.00,critique_brainz;pitchfork,2.0,2007-02-23,...,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,290043,<NA>
4,4,!!!,Shake the Shudder,1,Rock,3.65,3.65,pitchfork,1.0,2017-05-24,...,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,73665,<NA>


In [37]:
out_albums.sample(5)

,album_id,artist,album,review_count,genre_pitchfork,rating_mean,rating_pf,sources,n_sources,published_on_min,...,release_year,release_type,artist_type,artist_area,label,catalog_number,mb_tags,mb_genre,lastfm_listeners,lastfm_tags
15957,16058,Menace Ruine,Venus Armata,1,NaN,3.7,3.7,pitchfork,1.0,2014-10-22,...,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,<NA>,<NA>
10947,11070,Is Tropical,Native To,2,"Electronic, Rock",2.5,2.5,critique_brainz;pitchfork,2.0,2011-07-07,...,2011.0,Album,Group,United Kingdom,Kitsuné,CDA035,indie rock;rock,Rock,<NA>,<NA>
5998,5908,David Holmes,The Holy Pictures,2,Electronic,3.9,3.9,critique_brainz;pitchfork,2.0,2008-09-05,...,2008.0,Album,Person,United Kingdom,Mercury Records,1777275,psychedelic rock;indie rock;electronic;experim...,Rock,55878,electronic;shoegaze;2008;totec radio;skins
13539,13650,La Big Vic,Dub the World! Actually Remixed,1,Pop/R&B,3.7,3.7,pitchfork,1.0,2012-02-21,...,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,NaN,<NA>,<NA>
21766,21934,Shirley Bassey,The Greatest Hits: This Is My Life,1,NaN,NaN,NaN,critique_brainz,1.0,2009-09-18,...,2000.0,Album,Person,Monte-Carlo,Liberty EMI Records UK,72435-31050-2-0,easy listening;latin;pop;soul,Easy Listening/Vocal,<NA>,<NA>


## Join onto embedding recommendations

Adds `query_*` and `rec_*` Last.fm fields to each parquet row via the enriched catalog lookup. Rows whose artist/album names are not in `albums.csv` stay null (common for RA-only albums in the parquet).

In [ ]:
lookup = albums.set_index(["artist_norm", "album_norm"])[[LISTENER_COL, TAGS_COL]]

def attach_lastfm(frame, artist_col, album_col, prefix):
    keys = pd.MultiIndex.from_arrays(
        [frame[artist_col].map(normalize), frame[album_col].map(normalize)]
    )
    matched = lookup.reindex(keys)
    out = frame.copy()
    out[f"{prefix}_{LISTENER_COL}"] = matched[LISTENER_COL].to_numpy()
    out[f"{prefix}_{TAGS_COL}"] = matched[TAGS_COL].to_numpy()
    return out


recs = pd.read_parquet(RECS_PATH)
recs = attach_lastfm(recs, "query_artist", "query_album", "query")
recs = attach_lastfm(recs, "rec_artist", "rec_album", "rec")

for side in ("query", "rec"):
    col = f"{side}_{LISTENER_COL}"
    print(f"{col}: {recs[col].notna().mean():.1%} of rows")

recs.to_parquet(RECS_OUT_PATH, index=False)
print(f"Saved {RECS_OUT_PATH}")
recs.head()